### Olist Bronze Ingestion
This notebook ingests the original Olist CSV files into Delta tables without applying business transformations.

In [0]:
olist_path = '/Volumes/workspace/ecommerce_bronze/raw_files/olist/olistecommerce'

display(dbutils.fs.ls(olist_path))

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType

CATALOG = "workspace"
BRONZE_SCHEMA = "ecommerce_bronze"
VOLUME_PATH = (
    f"/Volumes/{CATALOG}/"
    f"{BRONZE_SCHEMA}/raw_files/olist/olistecommerce"
)



In [0]:
#Read, display and count the orders

orders_path = f"{VOLUME_PATH}/olist_orders_dataset.csv"
orders_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(orders_path)
)

display(orders_df.limit(10))
orders_df.printSchema()
orders_df.count()

In [0]:
#Added technical metadata

orders_bronze_df = (
    orders_df
    .withColumn(
            "_ingested_at",
            F.current_timestamp(),
        )
        .withColumn(
            "_source_file_path",
            F.col("_metadata.file_path"),
        )
        .withColumn(
            "_source_file_name",
            F.col("_metadata.file_name"),
        )
) 



In [0]:
#Saved as a Delta table
(
    orders_bronze_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", True)
    .saveAsTable(
        f"{CATALOG}.{BRONZE_SCHEMA}.olist_orders"
    )
)

In [0]:
%sql
SELECT *
FROM workspace.ecommerce_bronze.olist_orders
LIMIT 50;

In [0]:
%sql
SELECT COUNT(*) AS total_orders
FROM workspace.ecommerce_bronze.olist_orders

In [0]:
#Ingestion function

def ingest_csv_to_delta(
    file_name: str,
    table_name: str,
) -> None: 
    """
    Reads a CSV file from the Olist volume and writes it as a Bronze Delta table.
    """

    source_path = f"{VOLUME_PATH}/{file_name}"
    target_table = f"{CATALOG}.{BRONZE_SCHEMA}.{table_name}"

    dataframe = (
        spark.read
        .option("header", True)
        .option("inferSchema", True)
        .csv(source_path)
        .withColumn("_ingested_at", F.current_timestamp())
        .withColumn("_source_file_name", F.lit(file_name))
    )
    (
        dataframe.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", True)
        .saveAsTable(target_table)
    )
    row_count = dataframe.count()

    print(f"Created: {target_table}")
    print(f"Rows: {row_count:,}")


In [0]:
#Dictionary

olist_files = {
    "olist_customers_dataset.csv": "olist_customers",
    "olist_geolocation_dataset.csv": "olist_geolocation",
    "olist_order_items_dataset.csv": "olist_order_items",
    "olist_order_payments_dataset.csv": "olist_order_payments",
    "olist_order_reviews_dataset.csv": "olist_order_reviews",
    "olist_orders_dataset.csv": "olist_orders",
    "olist_products_dataset.csv": "olist_products",
    "olist_sellers_dataset.csv": "olist_sellers",
    "product_category_name_translation.csv": "product_category_translation",
}

for file_name, table_name in olist_files.items():
    ingest_csv_to_delta(file_name, table_name)


In [0]:
%sql
SHOW TABLES IN workspace.ecommerce_bronze

In [0]:
#Create Bronze table/report
bronze_table_names = list(olist_files.values())
table_summary = []

for table_name in bronze_table_names:
    full_table_name = f"{CATALOG}.{BRONZE_SCHEMA}.{table_name}"
    dataframe = spark.table(full_table_name)

    table_summary.append(
        {
        "table_name": table_name,
        "row_count": dataframe.count(),
        "column_count": len(dataframe.columns),

        }
    )
summary_df = spark.createDataFrame(table_summary)
display(summary_df.orderBy("table_name"))


In [0]:
#Save the table

(
    summary_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        f"{CATALOG}.{BRONZE_SCHEMA}.ingestion_summary"
    )
)